# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- Lecture 01: For loop in Task 1.2 and 5.1
- Lecture 02: Append list in Task 1.2, define function in Task 1.2, F string in numpy checkpoint, summarizing collection (e.g. min/max/len)
- Lecture 03: Output for Task 4.3:
    shape: (27149, 2)
    [143.01337066 137.15061328]
    The baseline mean is 143.01337066 and the week12 mean is 137.15061328. I used axis = 0 because we are calculating mean per visit across patients to get the baseline and week12 mean so we are looking between rows (vertically).
- Lecture 04: Select rows in Task 4.2, unique values in Task 5.1, read_csv in Task 5.3
- Lecture 05: Rules in Task 2.3, cleaning in Task 3.1, 3.2, recording decision in Task 3.3, validation in Task 5.1
- Lecture 06: Concatenation in Task 2.2, reshaping from wide to long in Task 4.1, merge in Task 4.2, 4.4, reshaping from long to wide in Task 4.3

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    """Return name, rows, columns, and number of duplicate rows in the dataset"""
    return {
    "name": Path(name).name,
    "rows": table.shape[0],
    "columns": table.shape[1],
    "exact_duplicate_rows": table.duplicated().sum()
    }


results = []

for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

results = pd.DataFrame(results)

results.to_csv(PROFILE_PATH, index=False)


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

patient: one patient; "patient_id"; extra repeats
visit: one patient's visit (combination of visit_north and visit_south); "patient_id", "visit"; extra repeats
a1c_wide: one patient's a1c record across visits; "patient_id"; wide format - too many visit for one record, see next line
a1c_long: one patient's a1c record for one visit ; "patient_id", "visit"; long format
site: one current site; "site_code"; contains both current and retired records

the cardinality you expect for each Task 4.4 join:
visit_clean merge with patients_clean on patient_id: many to one (repeated left keys), one patient can have multiple visits
visit_clean merge with sites_current on site_code: many to one (repeated left keys), one site can have multiple visits
visit_clean merge with a1c_long on ["patient_id", "visit"]: one to one (unique keys on both side), one patient have one unique a1c record


### 2.2 Stack the two visit files

In [4]:
visits_south = raw["visits_south"].copy(deep=True).rename(columns= {'weight_kg': 'wt_kg'})
visits_north = raw["visits_north"].copy(deep=True)

visits_north['source_file'] = "visits_north"
visits_south['source_file'] = "visits_south"

visits_raw = pd.concat([visits_north, visits_south], ignore_index=True)

### 2.3 Audit the raw tables

In [5]:
issue_audit = {
"patients: gender values that are not a code": (raw["patients"]["gender"].str.strip().notna() & raw["patients"]['gender'].str.strip().ne("") & ~raw["patients"]['gender'].str.strip().isin(['F', 'M', 'X'])).sum(),
"patients: empty gender": (raw["patients"]["gender"].str.strip().isna() | raw["patients"]["gender"].str.strip().eq("")).sum(),
"patients: birth_year sentinel 9999": pd.to_numeric(raw["patients"]["birth_year"], errors = "coerce").eq(9999).sum(),
"patients: site codes needing normalization": raw["patients"]["site_code"].str.strip().str.upper().str.replace("-", "").str.replace(" ", "").ne(raw["patients"]["site_code"]).sum(),
"sites: rows with a repeated site_code": raw["sites"].duplicated(subset=["site_code"], keep=False).sum(),
"visits: visit labels that are not baseline or week12": (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
"visits: sbp sentinel 999": pd.to_numeric(visits_raw["sbp_mmhg"], errors = "coerce").eq(999).sum(),
"visits: sbp outside 70 through 250": (pd.to_numeric(visits_raw["sbp_mmhg"], errors = "coerce").ne(999) & ~pd.to_numeric(visits_raw["sbp_mmhg"], errors = "coerce").between(70, 250)).sum(),
"visits: date parse failures": pd.to_datetime(visits_raw["visit_date"].where(visits_raw["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}', na = False)), format='%Y-%m-%d', errors='coerce').isna().sum(),
"visits: rows in repeated patient_id and visit pairs": (visits_raw["visit"].str.lower().isin(["baseline", "week12"]) & visits_raw.assign(visit = visits_raw["visit"].str.lower()).duplicated(subset=["patient_id", "visit"], keep=False)).sum()
}

issue_audit = pd.DataFrame(issue_audit.items(), columns = ["issue", "count"])

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

issue_audit.to_csv(AUDIT_PATH, index=False)

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates()

patients_clean['gender'].value_counts(dropna=False)

gender_codes = {
    'f': 'F',
    'female': 'F',
    'woman': 'F',
    'm': 'M',
    'male': 'M',
    'man': 'M',
    'x': 'X',
    'nb': 'X',
    'nonbinary': 'X',
    'non-binary': 'X'
}

patients_clean['gender'] = patients_clean['gender'].str.strip().str.lower().map(gender_codes)

patients_clean['gender'] = patients_clean['gender'].astype("category")

patients_clean['birth_year'] = patients_clean['birth_year'].replace("9999", np.nan).astype('Int64')

patients_clean["site_code"] = patients_clean["site_code"].str.strip().str.upper().str.replace("-", "").str.replace(" ", "")

patients_clean = patients_clean.reset_index(drop=True)

patients_summary = {
"patients": len(patients_clean),
"birth_year missing": patients_clean["birth_year"].isna().sum(),
"birth_year min": patients_clean["birth_year"].min(),
"birth_year max": patients_clean["birth_year"].max(),
"site_code values": patients_clean["site_code"].nunique(),
"patients at N01": (patients_clean["site_code"] == "N01").sum(),
"patients at N02": (patients_clean["site_code"] == "S02").sum(),
"patients at N04": (patients_clean["site_code"] == "E04").sum(),
}

patients_summary = pd.DataFrame(patients_summary.items(), columns = ["metric", "value"])

In [7]:
patients_clean["gender"].cat.categories

patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True)

visits_clean = visits_clean.drop_duplicates()

visits_clean['visit'].value_counts(dropna=False)

visit_codes = {
    'baseline': 'baseline',
    'bl': 'baseline',
    'week12': 'week12',
    'wk12': 'week12',
    'week 12': 'week12'
}

visits_clean['visit'] = visits_clean['visit'].str.strip().str.lower().map(visit_codes)

visits_clean['sbp_mmhg'] = pd.to_numeric(visits_clean['sbp_mmhg'], errors = "coerce")
visits_clean['sbp_mmhg'] = visits_clean['sbp_mmhg'].where(visits_clean['sbp_mmhg'].between(70, 250)).astype('Int64')

visits_clean['weight_kg'] = pd.to_numeric(visits_clean['weight_kg'], errors = "coerce")

exact = visits_clean['visit_date'].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
visits_clean['visit_date'] = pd.to_datetime(visits_clean['visit_date'].where(exact), format='%Y-%m-%d', errors='coerce')

In [9]:
visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"])]
visits_clean = visits_clean.sort_values("visit_time")
visits_clean = visits_clean.drop_duplicates(subset=["patient_id", "visit"], keep="last")

visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[[
    "patient_id",
    "visit",
    "visit_date",
    "sbp_mmhg",
    "weight_kg",
    "source_file"
]]

In [10]:
visits_summary = {
"visits": len(visits_clean),
"baseline visits": (visits_clean["visit"] == "baseline").sum(),
"week12 visits": (visits_clean["visit"] == "week12").sum(),
"visit_date missing": visits_clean["visit_date"].isna().sum(),
"visit_date min": visits_clean["visit_date"].min(),
"visit_date max": visits_clean["visit_date"].max(),
"sbp_mmhg missing": visits_clean["sbp_mmhg"].isna().sum(),
"sbp_mmhg min": visits_clean["sbp_mmhg"].min(),
"sbp_mmhg max": visits_clean["sbp_mmhg"].max(),
"weight_kg missing": visits_clean["weight_kg"].isna().sum(),
"weight_kg mean": visits_clean["weight_kg"].mean(),
"visits from visits_north.csv": (visits_clean["source_file"] == "visits_north").sum(),
"visits from visits_south.csv": (visits_clean["source_file"] == "visits_south").sum(),
}

visits_summary = pd.DataFrame(visits_summary.items(), columns = ["metric", "value"])

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame({
    'table': ['patients', 'patients', 'patients', 'patients', 'visits', 'visits', 'visits', 'visits', 'sites'],
    'issue': ['exact duplicate rows', 'gender spellings that are not a code', 'site codes needing normalization', 'birth_year sentinel 9999',
              'weight column named wt_kg in visits_south.csv', 'sbp sentinel 999 and readings outside 70 through 250', 'impossible and loosely written visit dates',
              'repeated patient_id and visit pairs', 'repeated site_code rows'],
    'decision': ['keep the first raw copy', 'correct to codes only', 'strip and lowercase', 'set to missing',
               'rename weight column', 'set to missing', 'set to missing', 'keep the first row', 'keep the first row'],
    'reason': ['identical in every raw column', 'M/F/NB only for convenience',
               'same site code typed differently', 'birth year 9999 is missing',
               'to match the other csv weight column', 'sbp 999 is missing and lower than 70 or over 250 is impossible',
               'probably typo in dates', 'identical in the extra record', 'identical in the extra record'],
})
display(decision_table)

decision_table.to_csv(DECISION_LOG_PATH, index=False)

,table,issue,decision,reason
0,patients,exact duplicate rows,keep the first raw copy,identical in every raw column
1,patients,gender spellings that are not a code,correct to codes only,M/F/NB only for convenience
2,patients,site codes needing normalization,strip and lowercase,same site code typed differently
3,patients,birth_year sentinel 9999,set to missing,birth year 9999 is missing
4,visits,weight column named wt_kg in visits_south.csv,rename weight column,to match the other csv weight column
5,visits,sbp sentinel 999 and readings outside 70 throu...,set to missing,sbp 999 is missing and lower than 70 or over 2...
6,visits,impossible and loosely written visit dates,set to missing,probably typo in dates
7,visits,repeated patient_id and visit pairs,keep the first row,identical in the extra record
8,sites,repeated site_code rows,keep the first row,identical in the extra record


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct",
)
print(a1c_long.shape)
display(a1c_long.head(6))

print(a1c_long['visit'].value_counts(dropna=False))
visit_codes_2 = {
    'a1c_baseline': 'baseline',
    'a1c_week12': 'week12'
}
a1c_long['visit'] = a1c_long['visit'].str.strip().str.lower().map(visit_codes_2)

a1c_long["a1c_pct"].dtype

a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
"visits": ["baseline", "week12"],
"results": [
    (a1c_long["visit"] == "baseline").sum(),
    (a1c_long["visit"] == "week12").sum()
    ],
"a1c_pct_mean": [
    round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2),
    round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2)
]
})
display(a1c_summary)

a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

(58848, 3)


,patient_id,visit,a1c_pct
0,P00001,a1c_baseline,7.3
1,P00002,a1c_baseline,8.4
2,P00003,a1c_baseline,5.9
3,P00004,a1c_baseline,7.3
4,P00005,a1c_baseline,7.9
5,P00006,a1c_baseline,NaN


visit
a1c_baseline    29424
a1c_week12      29424
Name: count, dtype: int64


,visits,results,a1c_pct_mean
0,baseline,29424,8.15
1,week12,29424,7.63


### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]
try:
    pd.merge(patients_clean, sites, on='site_code', how='left', validate='many_to_one')

except pd.errors.MergeError as error:
    print('MergeError:', error)

print(sites[sites.duplicated(subset=["site_code"], keep=False)])

sites_current = sites.loc[sites['record_status'] == 'current', ["site_code", "site_name"]].reset_index(drop=True)

display(sites_current)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...
  site_code            site_name record_status
0       N01         North Clinic       current
1       N01  North Street Clinic       retired
2       S02     Southgate Clinic       retired
3       S02  South Valley Clinic       current


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


In [14]:
merge = pd.merge(visits_clean, a1c_long, on=["patient_id", "visit"], how='outer', indicator=True)

display(merge['_merge'].value_counts())

merge_audit = {
"visits with an A1c result": (merge["_merge"] == "both").sum(),
"visits without an A1c result": (merge["_merge"] == "left_only").sum(),
"A1c results without a visit": (merge["_merge"] == "right_only").sum(),
"visits whose patient is not in the registry": (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
"registry patients with no visit": (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
"registry patients whose site is not in the site list": (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
"site list rows removed by the current-record rule": len(sites) - len(sites_current)
}

merge_audit = pd.DataFrame(merge_audit.items(), columns = ["check", "count"])

merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)

_merge
both          55945
right_only     2903
left_only      1236
Name: count, dtype: int64

### 4.3 Pivot blood pressure to one row per patient

In [15]:
sbp_wide = visits_clean.pivot(
    index="patient_id", 
    columns="visit", 
    values="sbp_mmhg"
).reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = (sbp_wide["week12"] - sbp_wide["baseline"])

sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].to_csv(SBP_WIDE_SAMPLE_PATH, index=False)

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
complete_pairs = (sbp_wide[["baseline", "week12"]].dropna())

complete_pairs = np.array(complete_pairs, dtype=float)

array_shape = f"shape: {complete_pairs.shape}"
print(array_shape)

mean_across_patients = np.mean(complete_pairs, axis = 0)
print(mean_across_patients)

shape: (27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [17]:
analysis = pd.merge(visits_clean, patients_clean, on='patient_id', how='left', validate='many_to_one')
analysis = pd.merge(analysis, sites_current, on='site_code', how='left', validate='many_to_one')
analysis = pd.merge(analysis, a1c_long, on=["patient_id", "visit"], how='left', validate='one_to_one')

analysis = analysis.astype({
    "patient_id": "string", 
    "visit_date": "datetime64[us]", 
    "sbp_mmhg": "Int64", 
    "weight_kg": "float64", 
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string"
    })

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
checks = pd.Series({
    'as many rows': len(analysis) == len(visits_clean),
    'no id or visit repeats': ~(analysis.duplicated(subset = ['patient_id', 'visit']).any()),
    'sbpmmHg in range': analysis['sbp_mmhg'].dropna().between(70, 250).all(),
    'site is current': analysis['site_name'].dropna().isin(sites_current['site_name']).all(),
    'sbp_wide["patient_id"] is unique': sbp_wide["patient_id"].is_unique,
    'raw table match snapshot': all(raw[name].equals(raw_snapshot[name]) for name in raw)
})

assert checks.all(), checks[~checks]
print(checks[~checks])

Series([], dtype: bool)


### 5.2 Save the analysis extract

In [19]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

### 5.3 Read the file back

In [20]:
round_trip = pd.read_csv("output/analysis_sample.csv",
    dtype = {
        "patient_id": "string", 
        "sbp_mmhg": "Int64", 
        "weight_kg": "float64", 
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string"
        },
        parse_dates=["visit_date"]
)
print(round_trip.equals(analysis_sample))

True
